In [2]:
# Uncomment in case packages are not installed
!pip install timm
!pip install transformers
!pip install pyspellchecker
!pip install torch torchvision
!pip install ftfy
!pip install regex
!pip install sentencepiece

In [4]:
import torch
from torch import nn
import argparse
import numpy as np
import json

torch.cuda.empty_cache()
import pandas as pd
import re

import os
import cv2
import gc
import itertools
from tqdm.autonotebook import tqdm
import albumentations as A


import torch.nn.functional as F
import timm

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import spacy
#For displaying loadbar
from tqdm import tqdm
import operator
from PIL import Image

# This is our visual library
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
import math
import seaborn as sns
from wordcloud import WordCloud
from wordcloud import ImageColorGenerator

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.metrics import confusion_matrix

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /usr/share/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package names to /usr/share/nltk_data...
[nltk_data]   Package names is already up-to-date!


/tmp/ipykernel_25/3378561779.py:50: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn-whitegrid')


# Initialization

In [5]:
class CFG:
    # If there's a GPU available...
    if torch.cuda.is_available():
      # Tell PyTorch to use the GPU.
      device = torch.device("cuda")
      print('There are %d GPU(s) available.' % torch.cuda.device_count())
      print('We will use the GPU:', torch.cuda.get_device_name())
    # If not...
    else:
      print('No GPU available, using the CPU instead.')
      device = torch.device("cpu")

    image_path = "/kaggle/input/exist-memes-dataset/EXIST 2024 Memes Dataset/EXIST 2024 Memes Dataset/training/memes/"
    train_file = '/kaggle/input/exist-dataset-folds/fold_0_train.csv'
    test_file = '/kaggle/input/exist-dataset-folds/fold_0_val.csv'



There are 1 GPU(s) available.
We will use the GPU: Tesla P100-PCIE-16GB


In [7]:
# Count the number of samples with 0 and 1 in the 'label' column
df_train = pd.read_csv(CFG.train_file, header='infer', keep_default_na=False)
df_train = df_train.sample(frac=1).reset_index(drop=True)

# Drop rows where 'value' is either NaN or an empty string
df_train = df_train[~df_train['value'].isin(['', None])]
df_train = df_train.reset_index(drop=True)

# Verify the change
print(df_train['value'].value_counts())

df_train['value'] = df_train['value'].map({'YES': 1, 'NO': 0}).fillna(df_train['value']).astype(int)

df_val = pd.read_csv(CFG.test_file, header='infer', keep_default_na=False)

# Drop rows where 'value' is either NaN or an empty string
df_val = df_val[~df_val['value'].isin(['', None])]

# Reset the index after dropping rows
df_val = df_val.reset_index(drop=True)

df_val['value'] = df_val['value'].map({'YES': 1, 'NO': 0}).fillna(df_val['value']).astype(int)

# Check the updated dataframe

print(df_val['value'].value_counts())

train_image_names = df_train['meme']
train_image_path = []
for name in train_image_names:
  train_image_path.append(CFG.image_path + name)

train_texts =  df_train['text']
train_texts = train_texts.astype(str)
train_labels =  df_train['value']

val_image_names = df_val['meme']
val_image_path = []
for name in val_image_names:
  val_image_path.append(CFG.image_path + name)

val_texts =  df_val['text']
val_texts = val_texts.astype(str)
test_labels =  df_val['value']


value
YES    1833
NO     1243
Name: count, dtype: int64
value
1    205
0    139
Name: count, dtype: int64


# BLIP Model

In [6]:
from transformers import BlipProcessor, BlipForConditionalGeneration, Trainer, TrainingArguments

class MemeDataset(Dataset):
    def __init__(self, dataframe, image_dir, processor, max_length=128):
        self.data = dataframe
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.data.iloc[idx]["meme"])
        image = Image.open(img_path).convert("RGB")
        text = self.data.iloc[idx]["text"]
        label = self.data.iloc[idx]["value"]

        # Process image and text
        inputs = self.processor(images=image, text=text, return_tensors="pt", padding="max_length", max_length=self.max_length, truncation=True)
        inputs["labels"] = torch.tensor(label, dtype=torch.long)
        
        return {key: val.squeeze(0) for key, val in inputs.items()}

class BlipForPartialFineTuning(nn.Module):
    def __init__(self, blip_model, num_labels=2, unfreeze_last_n_layers=1):
        super(BlipForPartialFineTuning, self).__init__()
        self.blip_model = blip_model

        # Use a dummy input to retrieve the hidden size
        sample_pixel_values = torch.rand(1, 3, 224, 224)  # Adjust shape based on your input image size
        sample_input_ids = torch.randint(0, 10, (1, 10))  # Example token IDs; adjust length as needed
        sample_output = self.blip_model(pixel_values=sample_pixel_values, input_ids=sample_input_ids, return_dict=True)
        hidden_size = sample_output.last_hidden_state.shape[-1]
        
        # Define the classification layer
        self.classifier = nn.Linear(hidden_size, num_labels)
        
        # Freeze all layers in the BLIP model
        for param in self.blip_model.parameters():
            param.requires_grad = False

        # Unfreeze the last few layers based on the user-specified value
        for layer in self.blip_model.vision_model.encoder.layers[-unfreeze_last_n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

    def forward(self, pixel_values, input_ids, attention_mask=None, labels=None):
        # Get image-text features
        outputs = self.blip_model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # Use CLS token or other pooling strategy
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


# Initialize processor and model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForPartialFineTuning(blip_model, unfreeze_last_n_layers=3)  # Adjust the number of layers to unfreeze

# Load dataset
train_dataset = MemeDataset(dataframe=df_train, image_dir=CFG.image_path, processor=processor)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [7]:

# Set training arguments
training_args = TrainingArguments(
    output_dir="no",
    evaluation_strategy="no",
    per_device_train_batch_size=8,
    # per_device_eval_batch_size=8,
    num_train_epochs=10,
    save_strategy="no",
    logging_dir="./logs",
    report_to="none",  # Disable W&B logging
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    # eval_dataset=eval_dataset,
)

# Train model
trainer.train()

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss
500,0.653900
1000,0.484400
1500,0.262500
2000,0.097900
2500,0.022800
3000,0.009000
3500,0.009400


TrainOutput(global_step=3850, training_loss=0.2005171037958814, metrics={'train_runtime': 1959.9237, 'train_samples_per_second': 15.7, 'train_steps_per_second': 1.964, 'total_flos': 0.0, 'train_loss': 0.2005171037958814, 'epoch': 10.0})

In [8]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# # Assuming you already have a MemeDataset for test data
# test_dataset = MemeDataset(csv_file="test.csv", image_dir="path/to/test/images", processor=processor)
eval_dataset = MemeDataset(dataframe=df_val, image_dir=CFG.image_path, processor=processor)

# Define a custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    all_predictions = torch.argmax(torch.tensor(logits), dim=-1)
    print("Predictions:", all_predictions)

    target_names = ['0', '1']
    classification_matrix = classification_report(test_labels, all_predictions, target_names=target_names)
    
    cm = confusion_matrix(test_labels, all_predictions)
    cm_df = pd.DataFrame(cm, index=['True_0', 'True_1'], columns=['Pred_0', 'Pred_1'])
        
    auc = roc_auc_score(test_labels, all_predictions)
    f1s = f1_score(test_labels, all_predictions)

    # Convert to a DataFrame
    df = pd.DataFrame({"Predicted Labels": all_predictions})
    
    # Save to CSV
    df.to_csv("/kaggle/working/Dis_blip_exist_10_fold5.csv", index=False)

    return {
        "Confusion Matrix": cm_df,
        "Classification Matrix": classification_matrix,
        "AUC": auc,
        "f1 score": f1s,
    }

# Initialize the Trainer with compute_metrics for evaluation
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics
)

# Evaluate the model on the test dataset
test_results = trainer.evaluate(eval_dataset)

# Print evaluation metrics
print("Confusion Matrix:", test_results["eval_Confusion Matrix"])
print("Classification Matrix:", test_results["eval_Classification Matrix"])
print("Area under Curve:", test_results["eval_AUC"])
print("F1 score:", test_results["eval_f1 score"])

Predictions: tensor([1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
        0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0,
        1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
        1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1,
        1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1,
        0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0,
        1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1,
        1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1,
        1, 1, 1, 1, 0, 0, 1